# Notebook 08: poisoning the normal model, and slow attacks

**Class 03_03.** An anomaly detector learns *normal* from data. If it is retrained on recent traffic (to follow
legitimate change), an attacker who can add traffic can teach it a new normal, a little at a time: the **boiling
frog** (Rubinstein et al. 2009 showed it on the PCA detector of backbone traffic; "Can't boil this frog", Kravchik
and Shabtai 2020, on online-trained autoencoders).

| | this notebook |
|:--|:--|
| goal | the target point (Example 1's exfiltration session) is accepted as normal after some rounds |
| capability | add a few points per round to the data the defender retrains on; the points must themselves pass |
| knowledge | the current model's score (grey box) |
| stage | training time (poisoning) |

Parts: A. boiling frog on Example 1 (Mahalanobis vs MCD); B. what the defender can watch; C. slow drift against
a per-point threshold, and CUSUM.

In [ ]:
import sys

import numpy as np
import polars as pl
from matplotlib import pyplot as plt

sys.path.insert(0, ".")
import plots_robust as pr

import anomlib as an
import deeplib as dl
import robustlib as rl

plt.rcParams["figure.dpi"] = 100
an.table_config()
X0 = an.example1(500, seed=0)
target = 2 * an.EX1_POINTS["B"]  # (2, -2): sends a lot, receives little
print("target d^2 under the clean model:", round(float(an.Mahalanobis().fit(X0).score(target[None])[0]), 1))

## A. The boiling frog

Each round the defender refits on its window of the last 500 accepted points and sets the threshold at 1% false
alarms. The attacker adds 25 points just below the current threshold on the segment from the normal centre to the
target. We follow the ratio score(target) / threshold: above 1 the target raises an alarm, below 1 it is accepted.

In [ ]:
runs = {
    "Mahalanobis (Ledoit-Wolf)": lambda: an.Mahalanobis(),
    "Mahalanobis (MCD, robust)": lambda: an.Mahalanobis(robust=True),
}
logs = {k: rl.boiling_frog(f, X0, target, rounds=20, per_round=25, keep=500, seed=0) for k, f in runs.items()}
plt.figure(figsize=(7, 3.5))
for k, log in logs.items():
    plt.plot([r["round"] for r in log], [r["target_ratio"] for r in log], "o-", label=k)
plt.axhline(1, color="k", ls="--", label="threshold")
plt.yscale("log")
plt.xlabel("poisoning round (25 points each)")
plt.ylabel("score(target) / threshold")
plt.legend()
plt.tight_layout()
plt.show()
rows = []
for k, log in logs.items():
    acc = next((r["round"] for r in log if r["target_ratio"] < 1), None)
    rows.append({"detector": k, "round accepted": acc, "poison points needed": None if acc is None else 25 * acc,
                 "share of the window": None if acc is None else min(1.0, 25 * acc / 500)})  # fmt: skip
print(pl.DataFrame(rows))
pr.boiling_frog(logs)  # slide plots: assets/figures/plots/03/

**Reading.** Both estimators accept the target after 8 or 9 rounds (200 to 225 poison points, 40-45% of the
window). The robust MCD does **not** help here: it discards *scattered* outliers (notebook 00, part C), but the poison
is a tight, patient cluster just inside the boundary, exactly the kind of dense region MCD keeps. Robust statistics
protect against noise and careless contamination, not against an attacker who adapts to the estimator.

## B. What the defender can watch

Poisoning must move the model, and a moving model is visible: track the centre of the training window between
retrains, and compare the newly accepted data with a frozen, trusted reference.

In [ ]:
ref = an.Mahalanobis().fit(X0)
t_ref = an.threshold(ref.score(X0))
plt.figure(figsize=(7, 3.5))
for k, log in logs.items():
    shift = [np.linalg.norm(r["centre"] - X0.mean(0)) for r in log]
    plt.plot(shift, "o-", label=k)
plt.xlabel("poisoning round")
plt.ylabel("shift of the training centre")
plt.legend()
plt.tight_layout()
plt.show()
print(
    f"frozen reference model: target ratio {float(ref.score(target[None])[0]) / t_ref:.1f} at every round (never poisoned, never adapts)"
)

A monotone walk of the centre in one direction is the signature of a boiling frog; legitimate change (a new
service, a holiday) rarely moves this steadily towards one place. Practical rules: retrain on data **vetted** by a
frozen reference model, cap how far one retrain may move the model, and keep the old model running in parallel.

## C. Slow drift: the per-point threshold never fires

Example 2 (the plant of notebook 06): the pressure sensor drifts slowly. A per-point detector (here the residual of
pressure predicted from the other sensors) compares each instant with a fixed threshold; a drift that adds a little
each step can stay under it for hours. **CUSUM** (Page 1954) accumulates small deviations:
$S_t=\max(0,\,S_{t-1}+(r_t-\mu)/\sigma-k)$, alarm when $S_t>h$.

In [ ]:
from sklearn.linear_model import LinearRegression

P = 7  # pressure


def hours_to_alarm(seed: int, slope_per_day: float) -> tuple[float | None, float | None]:
    """Fit on the first half of a plant run, inject the drift in the second half, hours until the first alarm."""
    X, _, _ = dl.plant(seed=seed)
    half = len(X) // 2
    X = (X - X[:half].mean(0)) / X[:half].std(0)
    reg = LinearRegression().fit(np.delete(X[:half], P, 1), X[:half, P])
    X[half:, P] += slope_per_day * np.arange(len(X) - half) / dl.DAY  # standard deviations per day
    r = X[:, P] - reg.predict(np.delete(X, P, 1))
    mu, sd = r[:half].mean(), r[:half].std()
    c = rl.cusum(r, mu, sd)
    t_point, h = np.abs(r[:half] - mu).max() * 1.1, c[:half].max() * 1.1  # same rule: above anything seen before
    first = lambda m: float(np.flatnonzero(m[half:])[0] / 12) if m[half:].any() else None  # noqa: E731  hours
    return first(np.abs(r - mu) > t_point), first(c > h)


rows = []
for slope in [0.1, 0.25, 0.5, 1.0, 2.0]:
    res = [hours_to_alarm(seed, slope) for seed in range(5)]
    med = lambda v: float(np.median([x if x is not None else np.inf for x in v]))  # noqa: E731
    rows.append({"drift (sd/day)": slope, "per-point: median hours": med([a for a, _ in res]),
                 "CUSUM: median hours": med([b for _, b in res]),
                 "per-point: missed (of 5)": sum(a is None for a, _ in res), "CUSUM: missed (of 5)": sum(b is None for _, b in res)})  # fmt: skip
print(pl.DataFrame(rows))
pr.slow_drift(rows)

Same false-alarm rule for both (above anything seen in the first half). The slower the drift, the longer a per-point
threshold waits (2.5 days at 0.1 sd/day, and one run never fires); CUSUM fires within 2 to 10 hours because it sums
the evidence.
The same idea defends against slow exfiltration (many small sessions) and against the boiling frog of part A:
test **accumulated** change, not only each point.